# Week 7 Exercise - Rithwik

### Fine-tuning an open source model

For Google Collab

In [ ]:
!pip install -q --upgrade transformers datasets peft trl bitsandbytes accelerate sentencepiece huggingface_hub 

In [ ]:
# imports
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

In [ ]:
# Constants

MODEL_ID        = "meta-llama/Llama-3.2-3B-Instruct"           # base model
OUTPUT_DIR      = "./llama3-alpaca-qlora"                       # local save path

# Dataset settings (match the GPT-4.1-nano notebook for easy comparison)
N_TRAIN_SAMPLES = 1000
N_VAL_SAMPLES   = 100
MAX_SEQ_LENGTH  = 512
RANDOM_SEED     = 42

# LoRA hyperparameters
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

# Training hyperparameters
NUM_EPOCHS      = 1
BATCH_SIZE      = 4
GRAD_ACCUM      = 4      # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE   = 2e-4

In [ ]:
# Log in to HuggingFace
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
#Load dataset
import random
random.seed(RANDOM_SEED)

print("Loading Alpaca dataset...")
raw_dataset = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Total examples: {len(raw_dataset)}")

# Quick look at a sample
sample = raw_dataset[42]
print(f"\nInstruction : {sample['instruction']}")
print(f"Input       : {sample['input']}")
print(f"Output      : {sample['output']}")

In [ ]:
# Curate and pre-process data
SYSTEM_PROMPT = (
    "You are a helpful, precise, and knowledgeable assistant. "
    "Answer questions clearly and follow instructions carefully."
)

def format_prompt(example: dict) -> dict:
    """
    Convert Alpaca row to Llama 3.2 chat template format.
    Uses the official <|begin_of_text|> instruction format.
    """
    instruction = example["instruction"].strip()
    inp         = example["input"].strip()
    output      = example["output"].strip()

    user_content = f"{instruction}\n\nContext:\n{inp}" if inp else instruction

    text = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{output}<|eot_id|>"
    )
    return {"text": text}

def is_quality(example: dict) -> bool:
    """Filter low-quality examples."""
    if len(example["instruction"].strip()) < 10: return False
    if len(example["output"].strip()) < 10:      return False
    if len(example["output"]) > 2000:             return False
    return True

# Filter, format, and subsample
clean    = raw_dataset.filter(is_quality)
clean    = clean.shuffle(seed=RANDOM_SEED)
clean    = clean.map(format_prompt)

train_dataset = clean.select(range(N_TRAIN_SAMPLES))
val_dataset   = clean.select(range(N_TRAIN_SAMPLES, N_TRAIN_SAMPLES + N_VAL_SAMPLES))

print(f"After filtering : {len(clean):,} examples")
print(f"Train split     : {len(train_dataset)}")
print(f"Val split       : {len(val_dataset)}")
print(f"\nFormatted example:")
print(train_dataset[0]["text"][:400], "...")

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size     : {tokenizer.vocab_size:,}")
print(f"Pad token      : {tokenizer.pad_token}")
print(f"EOS token      : {tokenizer.eos_token}")
print(f"Max length     : {tokenizer.model_max_length}")

In [ ]:
# Load Model & Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Llama 3.2 in 4-bit... (this may take few minutes)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",          # automatically distribute across available GPUs
    torch_dtype=torch.bfloat16,
    token=hf_token,
)
model.config.use_cache = False             # disable KV cache during training
model.config.pretraining_tp = 1

total_params = sum(p.numel() for p in model.parameters())
print(f"\n Model loaded")
print(f"   Total params : {total_params / 1e9:.2f}B")
# print(f"   Device map   : {model.hf_device_map}")

In [ ]:
#LoRA adapters
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # Target the attention projection layers — standard for Llama
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
total         = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of total)")
print(f"Total params     : {total:,}")
print(f"\nLoRA rank        : {LORA_R}")
print(f"LoRA alpha       : {LORA_ALPHA}")

In [ ]:
#Training setup
sft_config = SFTConfig(
    # Output
    output_dir=OUTPUT_DIR,

    # Training loop
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,           # trade speed for memory savings

    # Optimizer
    optim="paged_adamw_32bit",             # paged optimizer — essential for QLoRA
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,

    # Sequence
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,                          # set True to pack short sequences together

    # Precision
    fp16=False,
    bf16=False, # As T4 wasn't loading in GPU                              

    # Logging & evaluation
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",                       # set to "wandb" if desired

    seed=RANDOM_SEED,
)

print("Training config set")
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Learning rate        : {LEARNING_RATE}")

In [ ]:
# Run the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("\n Training complete!")

In [ ]:
# Visualizing
import matplotlib.pyplot as plt
import pandas as pd

log_history = trainer.state.log_history

train_logs = [{"step": x["step"], "loss": x["loss"]} 
              for x in log_history if "loss" in x]
eval_logs  = [{"step": x["step"], "eval_loss": x["eval_loss"]} 
              for x in log_history if "eval_loss" in x]

train_df = pd.DataFrame(train_logs)
eval_df  = pd.DataFrame(eval_logs)

plt.figure(figsize=(10, 4))
if not train_df.empty:
    plt.plot(train_df["step"], train_df["loss"], label="Train Loss", color="steelblue")
if not eval_df.empty:
    plt.plot(eval_df["step"], eval_df["eval_loss"], label="Val Loss", color="tomato")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Llama 3.2 QLoRA Fine-Tuning Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save the LoRA adapter weights
adapter_path = f"{OUTPUT_DIR}/adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f" LoRA adapters saved to: {adapter_path}")

# List saved files
import os
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f))
    print(f"   {f:40s}  {size / 1e6:.1f} MB")

In [ ]:
# Test the fine-tuned model
from transformers import pipeline

# Load base model + merge LoRA adapters for inference
print("Loading fine-tuned model for inference...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

gen_pipeline = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1,
)

def ask(instruction: str) -> str:
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{instruction}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )
    out = gen_pipeline(prompt)[0]["generated_text"]
    # Strip the prompt, return only the assistant response
    return out[len(prompt):].strip()

# Test prompts
test_prompts = [
    "Explain the difference between supervised and unsupervised learning in simple terms.",
    "Write a short poem about the ocean at sunset.",
    "What are three practical tips for improving focus while working from home?",
]

for prompt in test_prompts:
    print(f"USER  : {prompt}")
    print(f"MODEL : {ask(prompt)}")
    print("-" * 70)

In [ ]:
# # Push to Hugging Face
# HF_REPO = "mrithwik/llama3-alpaca-qlora"

# ft_model.push_to_hub(HF_REPO, token=hf_token)
# tokenizer.push_to_hub(HF_REPO, token=hf_token)
# print(f"Pushed to https://huggingface.co/{HF_REPO}")